# 7.4 — CNN Building Blocks

CNN building blocks are the small spatial choices that decide where local dot products land, how border pixels are treated, and when detail is thrown away. In this lesson, you will build stride, padding, pooling, and channel-aware shape calculations from scratch with NumPy so every output cell and tensor shape is inspectable.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build CNN blocks one idea at a time. Run each cell in order and read the printed intermediate values — every piece of shape arithmetic is spelled out so the stride, padding, pooling, and channel decisions are not black boxes. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays and hand-written CNN operations.
import matplotlib.pyplot as plt  # visualizations for maps, windows, and channels.
np.random.seed(0)  # reproducibility for any random examples.

### 1. A convolution response is still one local dot product

A CNN convolution begins with a small patch and a small kernel. The response at one location is the elementwise product of those two arrays followed by a sum: matching signs raise the response, opposite signs lower it, and zeros ignore positions. Stride, padding, and pooling do not change this local meaning; they change which patches are visited and how responses are kept.

In [ ]:
patch_w = np.array([[1., 2.],
                    [3., 4.]])  # one 2x2 image patch.
kernel_w = np.array([[1., 0.],
                     [0., -1.]])  # a tiny diagonal-difference detector.
products_w = patch_w * kernel_w  # elementwise contributions before summing.
print("patch:\n", patch_w)
print("kernel:\n", kernel_w)
print("elementwise products:\n", products_w)

▶ What you'll see: only the top-left and bottom-right entries contribute because the other kernel weights are zero.

In [ ]:
response_w = float(np.sum(products_w))  # local convolution response.
print("local response:", response_w)  # 1 + 0 + 0 - 4 = -3.
assert response_w == -3.0  # concrete lesson number.

▶ What you'll see: the response is `-3`, meaning this patch disagrees with the kernel's preferred diagonal pattern.

In [ ]:
plt.figure(figsize=(6, 2.5))
for idx_w, (M_w, title_w) in enumerate([(patch_w, "patch"), (kernel_w, "kernel"), (products_w, "products")]):
    plt.subplot(1, 3, idx_w + 1)
    plt.imshow(M_w, cmap="coolwarm")
    plt.title(title_w)
    plt.xticks([]); plt.yticks([])
    for r_w in range(M_w.shape[0]):
        for c_w in range(M_w.shape[1]):
            plt.text(c_w, r_w, f"{M_w[r_w, c_w]:.0f}", ha="center", va="center", color="black")
plt.suptitle("1: one convolution response = sum of products")
plt.show()

▶ What you'll see: the product panel shows exactly which cells add up to the scalar response.

*Why it's done this way:* convolution is a shared-weight dot product. Multiplying patch entries by kernel weights tests whether local evidence aligns with the pattern the kernel represents, and summing collapses that local agreement into one activation. This is why a CNN can reuse the same detector across many spatial positions.

### 2. Output size is a landing-position count

For a one-dimensional input length `n`, kernel size `k`, padding `p`, and stride `s`, the number of valid window landings is

$$n_{out}=\left\lfloor\frac{n+2p-k}{s}\right\rfloor+1.$$

The numerator `n + 2p - k` is the last legal start position after padding. Dividing by stride counts how many jumps fit, and the floor warns that leftover border positions are ignored rather than producing fractional cells.

In [ ]:
n_w, k_w, p_w, s_w = 5, 2, 0, 2  # 5-wide input, 2-wide kernel, no padding, stride 2.
last_start_w = n_w + 2 * p_w - k_w  # last legal starting index.
out_w = last_start_w // s_w + 1  # floor division implements the formula.
print("last legal start:", last_start_w)
print("output length:", out_w)
assert out_w == 2  # floor((5+0-2)/2)+1 = 2.

▶ What you'll see: the kernel can start at positions 0 and 2; the possible start at 3 is skipped by stride-2 landing.

In [ ]:
starts_w = np.arange(0, last_start_w + 1, s_w)  # actual landing positions.
all_starts_w = np.arange(0, last_start_w + 1)  # every possible stride-1 start.
print("all legal starts:", all_starts_w)
print("stride-2 starts:", starts_w)
assert np.array_equal(starts_w, np.array([0, 2]))  # concrete landing positions.

▶ What you'll see: stride chooses a subset of legal starts, not a fractional output position.

In [ ]:
plt.figure(figsize=(5, 1.8))
plt.scatter(all_starts_w, np.zeros_like(all_starts_w), s=120, color="lightgray", label="legal starts")
plt.scatter(starts_w, np.zeros_like(starts_w), s=180, color="seagreen", label="stride starts")
plt.yticks([]); plt.xticks(np.arange(5))
plt.title("2: output cells count window landings")
plt.legend(loc="upper right")
plt.show()

▶ What you'll see: only two green landings survive the stride schedule.

*Why it's done this way:* an output cell exists only when the whole kernel fits on the padded canvas. The floor is not a rounding convenience; it is the mathematical statement that incomplete final windows are dropped by standard convolution and pooling.

### 3. Stride shrinks by skipping patch starts

Stride changes the sampling grid. With a 5×5 image, 2×2 kernel, and stride 2, a hand-written convolution evaluates only four patches: top-left starts `(0,0)`, `(0,2)`, `(2,0)`, and `(2,2)`. The output is therefore 2×2 even though more overlapping stride-1 patches exist.

In [ ]:
X_w = np.arange(1, 26, dtype=float).reshape(5, 5)  # a 5x5 image with readable numbers.
K_w = np.array([[1., 0.],
                [0., -1.]])  # same local detector as before.
print("input shape:", X_w.shape)
print(X_w)

▶ What you'll see: a 5×5 grid whose values increase left-to-right and top-to-bottom.

In [ ]:
def conv2d_valid_w(X, K, stride=1):
    kh, kw = K.shape
    oh = (X.shape[0] - kh) // stride + 1
    ow = (X.shape[1] - kw) // stride + 1
    Y = np.zeros((oh, ow))
    for r in range(oh):
        for c in range(ow):
            patch = X[r * stride:r * stride + kh, c * stride:c * stride + kw]
            Y[r, c] = np.sum(patch * K)
    return Y

Y_stride_w = conv2d_valid_w(X_w, K_w, stride=2)  # evaluate every other landing.
print("stride-2 output:\n", Y_stride_w)
print("output shape:", Y_stride_w.shape)
assert Y_stride_w.shape == (2, 2)
assert np.all(Y_stride_w == -6.0)  # each 2x2 increasing patch has bottom-right minus top-left = 6.

▶ What you'll see: a 2×2 output; each response is `-6` because this detector subtracts the bottom-right value from the top-left value.

In [ ]:
plt.figure(figsize=(6, 2.6))
plt.subplot(1, 2, 1)
plt.imshow(X_w, cmap="viridis")
plt.title("input")
plt.xticks(range(5)); plt.yticks(range(5))
for r_w in [0, 2]:
    for c_w in [0, 2]:
        plt.scatter(c_w, r_w, color="red", s=80)
plt.subplot(1, 2, 2)
plt.imshow(Y_stride_w, cmap="magma")
plt.title("stride-2 responses")
plt.xticks(range(2)); plt.yticks(range(2))
plt.show()

▶ What you'll see: red dots mark the four top-left patch starts that become the four output cells.

*Why it's done this way:* stride is a deliberate resolution tradeoff. It reduces computation and feature-map size by evaluating fewer local dot products, but skipped starts mean the layer cannot represent every fine spatial shift in its output.

### 4. Padding gives border pixels more chances

Without padding, border pixels participate in fewer windows because kernels cannot hang outside the image. Padding creates a larger canvas by surrounding the image with zeros (or another chosen value), letting kernels land near the original border. The size formula shows that a 5×5 input with a 2×2 kernel, pad 1, and stride 1 grows to 6×6.

In [ ]:
X_pad_base_w = np.arange(1, 26, dtype=float).reshape(5, 5)
pad_w = 1
X_padded_w = np.pad(X_pad_base_w, pad_width=pad_w, mode="constant", constant_values=0)
print("original shape:", X_pad_base_w.shape, "padded shape:", X_padded_w.shape)
print(X_padded_w[:3, :3])
assert X_padded_w.shape == (7, 7)

▶ What you'll see: a zero border around the original 5×5 image.

In [ ]:
n_w, k_w, p_w, s_w = 5, 2, 1, 1
out_pad_w = (n_w + 2 * p_w - k_w) // s_w + 1
Y_pad_w = conv2d_valid_w(X_padded_w, K_w, stride=1)
print("formula output length:", out_pad_w)
print("actual padded-conv shape:", Y_pad_w.shape)
assert out_pad_w == 6
assert Y_pad_w.shape == (6, 6)

▶ What you'll see: padding enlarged the working canvas enough that a 2×2 kernel produces a 6×6 response map.

In [ ]:
plt.figure(figsize=(6, 2.7))
plt.subplot(1, 2, 1)
plt.imshow(X_padded_w, cmap="viridis")
plt.title("zero-padded input")
plt.xticks([]); plt.yticks([])
plt.subplot(1, 2, 2)
plt.imshow(Y_pad_w, cmap="magma")
plt.title("6x6 output")
plt.colorbar(fraction=0.046)
plt.xticks([]); plt.yticks([])
plt.show()

▶ What you'll see: the output is larger than the original image because an even-sized kernel plus pad 1 does not preserve size.

*Why it's done this way:* padding controls how much evidence borders receive. It is not automatically “same size”; the exact result depends on `n`, `k`, `p`, and `s`, and even kernels are especially easy to mis-size.

### 5. Pooling summarizes windows without learned weights

Pooling slides a window just like convolution, but replaces learned weighted sums with a fixed summary such as max or average. Max pooling keeps the strongest local activation, which is useful when the exact within-window location matters less than whether a feature appeared. Average pooling keeps the local mean, which is smoother but less selective.

In [ ]:
A_w = np.array([[1., 3., 2., 0.],
                [4., 6., 5., 1.],
                [1., 2., 9., 8.],
                [0., 1., 7., 3.]])  # activation map from the lesson.
print("activation map:\n", A_w)

▶ What you'll see: four 2×2 regions, each with a different strongest activation.

In [ ]:
def pool2d_w(X, size=2, stride=2, mode="max"):
    oh = (X.shape[0] - size) // stride + 1
    ow = (X.shape[1] - size) // stride + 1
    Y = np.zeros((oh, ow))
    for r in range(oh):
        for c in range(ow):
            window = X[r * stride:r * stride + size, c * stride:c * stride + size]
            Y[r, c] = np.max(window) if mode == "max" else np.mean(window)
    return Y

max_w = pool2d_w(A_w, size=2, stride=2, mode="max")
avg_w = pool2d_w(A_w, size=2, stride=2, mode="avg")
print("max pool:\n", max_w)
print("average pool:\n", np.round(avg_w, 2))
assert np.array_equal(max_w, np.array([[6., 5.], [2., 9.]]))

▶ What you'll see: max pooling returns `[[6, 5], [2, 9]]`, exactly the strongest response in each non-overlapping window.

In [ ]:
plt.figure(figsize=(6, 2.7))
plt.subplot(1, 2, 1)
plt.imshow(A_w, cmap="viridis")
plt.title("input activations")
plt.xticks(range(4)); plt.yticks(range(4))
plt.subplot(1, 2, 2)
plt.imshow(max_w, cmap="viridis")
plt.title("2x2 max pooled")
plt.xticks(range(2)); plt.yticks(range(2))
plt.show()

▶ What you'll see: the 4×4 map becomes 2×2, keeping each region's brightest evidence.

*Why it's done this way:* pooling is a cheap downsampler because it has no trainable parameters. The cost is information loss: max pooling discards all non-maximum values, while average pooling discards within-window arrangement.

### 6. Channels change parameter count, not the spatial formula

A real image has channels. A convolutional filter spans all input channels, so a 3×3 filter over RGB has `3×3×3 = 27` weights, not 9. The number of output channels is the number of filters. Spatial size still follows the same height/width formula independently of channel depth.

In [ ]:
height_w, width_w, in_channels_w = 32, 32, 3
kernel_h_w, kernel_w_w, out_channels_w = 3, 3, 8
pad_channels_w, stride_channels_w = 1, 1
spatial_out_w = (height_w + 2 * pad_channels_w - kernel_h_w) // stride_channels_w + 1
weights_per_filter_w = kernel_h_w * kernel_w_w * in_channels_w
total_weights_w = weights_per_filter_w * out_channels_w
print("spatial output:", spatial_out_w, "x", spatial_out_w)
print("weights per filter:", weights_per_filter_w)
print("total weights:", total_weights_w)
assert spatial_out_w == 32
assert weights_per_filter_w == 27
assert total_weights_w == 216

▶ What you'll see: padding preserves 32×32 spatial size, while eight filters create eight output channels.

In [ ]:
X_rgb_w = np.zeros((height_w, width_w, in_channels_w))
Y_shape_w = (spatial_out_w, spatial_out_w, out_channels_w)
print("input tensor shape:", X_rgb_w.shape)
print("output tensor shape:", Y_shape_w)
assert Y_shape_w == (32, 32, 8)

▶ What you'll see: the channel axis changes from 3 to 8 even though height and width stay at 32.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["spatial H", "spatial W", "input C", "output C"], [32, 32, 3, 8], color=["gray", "gray", "steelblue", "orange"])
plt.title("6: channels are separate from spatial size")
plt.ylabel("dimension size")
plt.show()

▶ What you'll see: height and width are preserved by padding, but filter count chooses the output depth.

*Why it's done this way:* every output channel is a different learned detector applied at every spatial location. Each detector must see all input channels to combine color or feature evidence, so channel depth multiplies parameters while the landing-position formula controls only height and width.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, shape arithmetic, and hand-written CNN operations.
import matplotlib.pyplot as plt # load Matplotlib for heatmaps and debugging plots.
np.random.seed(0) # make all examples reproducible.

def out_size(n, k, p=0, s=1): # compute one spatial output dimension for conv or pooling.
    return (n + 2 * p - k) // s + 1 # floor division counts valid window landings.

def conv2d(x, kernel, stride=1, pad=0): # implement single-channel 2-D convolution/cross-correlation from scratch.
    x = np.asarray(x, dtype=float) # ensure numeric array input.
    kernel = np.asarray(kernel, dtype=float) # ensure numeric kernel input.
    xp = np.pad(x, pad_width=pad, mode="constant", constant_values=0) # add zero padding around the spatial map.
    kh, kw = kernel.shape # read kernel height and width.
    oh = out_size(x.shape[0], kh, pad, stride) # compute output height from the shape formula.
    ow = out_size(x.shape[1], kw, pad, stride) # compute output width from the shape formula.
    y = np.zeros((oh, ow)) # allocate the output activation map.
    for r in range(oh): # loop over output rows.
        for c in range(ow): # loop over output columns.
            patch = xp[r * stride:r * stride + kh, c * stride:c * stride + kw] # select the input patch under the kernel.
            y[r, c] = np.sum(patch * kernel) # store the local dot-product response.
    return y # return the full response map.

def pool2d(x, size=2, stride=2, mode="max"): # implement non-learned pooling from scratch.
    x = np.asarray(x, dtype=float) # ensure numeric array input.
    oh = out_size(x.shape[0], size, 0, stride) # compute pooled height.
    ow = out_size(x.shape[1], size, 0, stride) # compute pooled width.
    y = np.zeros((oh, ow)) # allocate the pooled map.
    for r in range(oh): # loop over pooled rows.
        for c in range(ow): # loop over pooled columns.
            window = x[r * stride:r * stride + size, c * stride:c * stride + size] # select the pooling window.
            y[r, c] = np.max(window) if mode == "max" else np.mean(window) # summarize without learned weights.
    return y # return pooled activations.

def show_map(M, title): # define a compact heatmap helper.
    plt.figure(figsize=(4, 3)) # create a readable figure.
    plt.imshow(M, cmap="viridis", aspect="auto") # render values as colors.
    plt.colorbar(label="value") # add a numeric color scale.
    plt.title(title) # label the current map.
    plt.xlabel("column") # label horizontal spatial position.
    plt.ylabel("row") # label vertical spatial position.
    plt.show() # display the figure.

## 🟢 Basics (warm-up)

### Basic 1 — Compute one local response

**Goal.** Multiply a patch by a kernel and sum the products, because every convolution output cell starts as one local dot product. We build it in 2 steps.

In [ ]:
patch_b1 = np.array([[1., 2.], [3., 4.]]) # define one 2x2 patch.
kernel_b1 = np.array([[1., 0.], [0., -1.]]) # define a simple diagonal contrast kernel.
products_b1 = patch_b1 * kernel_b1 # compute per-position contributions.
print("products:\n", products_b1) # inspect the terms before summing.

▶ What you'll see: the kernel keeps `1` and `-4` while zeroing the other entries.

In [ ]:
response_b1 = float(np.sum(products_b1)) # sum contributions into one response.
print("response:", response_b1) # inspect the scalar activation.
assert response_b1 == -3.0 # verify the worked convolution response.
plt.figure(figsize=(4, 3)) # create a contribution plot.
plt.bar(["tl", "tr", "bl", "br"], products_b1.ravel(), color="teal") # show each contribution.
plt.title("Basic 1: local dot-product pieces") # title the plot.
plt.ylabel("patch × kernel") # label the contribution scale.
plt.show() # display the bar chart.

▶ What you'll see: the negative bottom-right contribution dominates the sum.

👀 Takeaway: a convolution response is a dot product between a local patch and shared kernel weights.

### Basic 2 — Count valid output landings

**Goal.** Apply the output-size formula in one dimension, because CNN shape bugs often begin with a wrong landing count. We build it in 2 steps.

In [ ]:
n_b2 = 5 # input length.
k_b2 = 2 # kernel length.
p_b2 = 0 # no padding.
s_b2 = 2 # stride two skips every other start.
last_start_b2 = n_b2 + 2 * p_b2 - k_b2 # compute the last legal starting position.
print("last start:", last_start_b2) # inspect the numerator of the formula.

▶ What you'll see: the last legal start is index 3, but stride 2 will not land on every legal start.

In [ ]:
out_b2 = out_size(n_b2, k_b2, p_b2, s_b2) # compute floor((n+2p-k)/s)+1.
starts_b2 = np.arange(0, last_start_b2 + 1, s_b2) # list actual stride starts.
print("output length:", out_b2, "starts:", starts_b2) # inspect output count and landings.
assert out_b2 == 2 # verify floor((5-2)/2)+1.
plt.figure(figsize=(4, 2)) # create a landing plot.
plt.scatter(starts_b2, np.zeros_like(starts_b2), s=160, color="seagreen") # draw valid stride landings.
plt.xticks(range(n_b2)); plt.yticks([]) # make input positions visible.
plt.title("Basic 2: stride landings") # title the plot.
plt.show() # display the landing positions.

▶ What you'll see: only two start positions become output cells.

👀 Takeaway: output size is a count of complete window landings, with leftover space dropped by the floor.

### Basic 3 — Convolve a tiny image with stride 1

**Goal.** Run a complete valid convolution by hand-coded loops, because the output map is just many local responses arranged on a grid. We build it in 2 steps.

In [ ]:
x_b3 = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]]) # define a small image.
k_b3 = np.array([[1., 0.], [0., -1.]]) # define the local detector.
y_b3 = conv2d(x_b3, k_b3, stride=1, pad=0) # compute all valid stride-1 responses.
print("output:\n", y_b3) # inspect the response map.

▶ What you'll see: a 2×2 output where each response compares a patch's top-left and bottom-right values.

In [ ]:
print("output shape:", y_b3.shape) # inspect the spatial shrinkage.
assert y_b3.shape == (2, 2) # 3x3 with 2x2 valid kernel gives 2x2.
assert np.all(y_b3 == -4.0) # every increasing 2x2 patch differs by 4 on the diagonal.
show_map(y_b3, "Basic 3: valid convolution output") # visualize the response map.

▶ What you'll see: every output cell has the same value because every 2×2 patch has the same diagonal gap.

👀 Takeaway: valid convolution shrinks spatial size when no padding is added.

### Basic 4 — Add zero padding before convolution

**Goal.** Pad an input before applying a kernel, because borders need extra canvas if they should receive more convolution landings. We build it in 2 steps.

In [ ]:
x_b4 = np.arange(1, 10, dtype=float).reshape(3, 3) # define a 3x3 image.
pad_b4 = 1 # add one border of zeros.
xp_b4 = np.pad(x_b4, pad_width=pad_b4, mode="constant", constant_values=0) # create the padded canvas.
print("padded shape:", xp_b4.shape) # inspect the enlarged canvas.
print(xp_b4) # inspect zero border placement.

▶ What you'll see: the original image is surrounded by a one-pixel zero frame.

In [ ]:
y_b4 = conv2d(x_b4, np.ones((3, 3)), stride=1, pad=pad_b4) # sum 3x3 neighborhoods with padding.
print("output shape:", y_b4.shape) # inspect output size.
assert y_b4.shape == (3, 3) # 3x3 kernel with pad 1 preserves a 3x3 map.
show_map(y_b4, "Basic 4: padded neighborhood sums") # visualize border effects.

▶ What you'll see: border sums are smaller because their padded windows include zeros.

👀 Takeaway: padding can preserve spatial size, but border responses still reflect the chosen padding values.

### Basic 5 — Use stride to downsample convolution

**Goal.** Compare stride 1 and stride 2 outputs, because stride reduces resolution by skipping window starts. We build it in 3 steps.

In [ ]:
x_b5 = np.arange(1, 26, dtype=float).reshape(5, 5) # create a readable 5x5 input.
k_b5 = np.ones((2, 2)) # use a 2x2 sum kernel.
y1_b5 = conv2d(x_b5, k_b5, stride=1, pad=0) # compute dense landings.
y2_b5 = conv2d(x_b5, k_b5, stride=2, pad=0) # compute skipped landings.
print("stride 1 shape:", y1_b5.shape, "stride 2 shape:", y2_b5.shape) # compare output sizes.

▶ What you'll see: stride 2 produces a smaller response map than stride 1.

In [ ]:
print("stride-2 output:\n", y2_b5) # inspect the downsampled responses.
assert y1_b5.shape == (4, 4) # 5x5 valid 2x2 stride 1 gives 4x4.
assert y2_b5.shape == (2, 2) # stride 2 gives 2x2.
assert y2_b5[0, 0] == 16.0 # 1+2+6+7 = 16.

▶ What you'll see: the first stride-2 response is the sum of the top-left 2×2 patch.

In [ ]:
plt.figure(figsize=(6, 2.7)) # create a shape comparison figure.
plt.subplot(1, 2, 1); plt.imshow(y1_b5, cmap="viridis"); plt.title("stride 1") # plot dense output.
plt.subplot(1, 2, 2); plt.imshow(y2_b5, cmap="viridis"); plt.title("stride 2") # plot skipped output.
plt.show() # display both maps.

▶ What you'll see: stride 2 keeps a coarser grid of the same local operation.

👀 Takeaway: stride changes where convolution is evaluated, not what each local response means.

### Basic 6 — Max pool a 4×4 map

**Goal.** Downsample an activation map with max pooling, because CNNs often keep the strongest local evidence while shrinking spatial size. We build it in 2 steps.

In [ ]:
a_b6 = np.array([[1., 3., 2., 0.], [4., 6., 5., 1.], [1., 2., 9., 8.], [0., 1., 7., 3.]]) # activation map.
max_b6 = pool2d(a_b6, size=2, stride=2, mode="max") # take max over non-overlapping 2x2 windows.
print("max pooled:\n", max_b6) # inspect the pooled map.

▶ What you'll see: max pooling returns `[[6, 5], [2, 9]]`.

In [ ]:
assert np.array_equal(max_b6, np.array([[6., 5.], [2., 9.]])) # verify the lesson pooling result.
show_map(max_b6, "Basic 6: 2x2 max pool") # visualize the pooled activations.

▶ What you'll see: the 4×4 map has been reduced to the strongest value in each quadrant.

👀 Takeaway: max pooling discards location within each window and keeps only the strongest activation.

### Basic 7 — Average pool the same windows

**Goal.** Compare average pooling with max pooling, because different summaries preserve different information from a window. We build it in 2 steps.

In [ ]:
a_b7 = np.array([[1., 3., 2., 0.], [4., 6., 5., 1.], [1., 2., 9., 8.], [0., 1., 7., 3.]]) # same activation map.
avg_b7 = pool2d(a_b7, size=2, stride=2, mode="avg") # average each 2x2 window.
print("average pooled:\n", np.round(avg_b7, 2)) # inspect smoothed downsampled values.

▶ What you'll see: average pooling returns smaller, smoother values than max pooling.

In [ ]:
assert np.allclose(avg_b7, np.array([[3.5, 2.0], [1.0, 6.75]])) # verify the four window means.
plt.figure(figsize=(5, 3)) # create a comparison chart.
plt.bar(["max top-left", "avg top-left"], [6.0, avg_b7[0, 0]], color=["orange", "steelblue"]) # compare one window summary.
plt.title("Basic 7: max vs average summary") # title the plot.
plt.ylabel("pooled value") # label pooled scale.
plt.show() # display the comparison.

▶ What you'll see: max keeps the peak 6, while average reports the window's overall level 3.5.

👀 Takeaway: average pooling is smoother; max pooling is more selective.

### Basic 8 — Track channel-aware filter weights

**Goal.** Count weights in a multi-channel convolution, because filters span every input channel. We build it in 2 steps.

In [ ]:
kh_b8 = 3 # kernel height.
kw_b8 = 3 # kernel width.
in_ch_b8 = 3 # RGB input channels.
out_ch_b8 = 8 # number of learned filters.
weights_per_filter_b8 = kh_b8 * kw_b8 * in_ch_b8 # count weights for one output channel.
print("weights per filter:", weights_per_filter_b8) # inspect 3x3x3.

▶ What you'll see: one RGB 3×3 filter has 27 weights.

In [ ]:
total_b8 = weights_per_filter_b8 * out_ch_b8 # multiply by the number of output filters.
print("total weights:", total_b8) # inspect total parameter count without biases.
assert weights_per_filter_b8 == 27 # verify channel depth is included.
assert total_b8 == 216 # verify eight output channels.
plt.figure(figsize=(4, 3)) # create a parameter-count plot.
plt.bar(["one filter", "8 filters"], [weights_per_filter_b8, total_b8], color=["teal", "purple"]) # compare counts.
plt.title("Basic 8: channel depth multiplies weights") # title the plot.
plt.ylabel("weight count") # label count axis.
plt.show() # display the bar chart.

▶ What you'll see: adding output channels multiplies the parameters linearly.

👀 Takeaway: spatial formulas ignore channels, but parameter counts cannot.

### Basic 9 — Compute a full output tensor shape

**Goal.** Combine spatial size and output-channel count, because a CNN layer returns a height×width×channels tensor. We build it in 2 steps.

In [ ]:
h_b9 = 32 # input height.
w_b9 = 32 # input width.
k_b9 = 3 # square kernel size.
p_b9 = 1 # same-size padding for odd 3x3 kernel.
s_b9 = 1 # stride 1.
out_ch_b9 = 8 # number of filters.
oh_b9 = out_size(h_b9, k_b9, p_b9, s_b9) # compute output height.
ow_b9 = out_size(w_b9, k_b9, p_b9, s_b9) # compute output width.
print("spatial output:", (oh_b9, ow_b9)) # inspect preserved spatial size.

▶ What you'll see: a 3×3 kernel with pad 1 and stride 1 preserves 32×32.

In [ ]:
shape_b9 = (oh_b9, ow_b9, out_ch_b9) # append output channels to spatial dimensions.
print("output tensor shape:", shape_b9) # inspect full layer output shape.
assert shape_b9 == (32, 32, 8) # verify the lesson shape.
plt.figure(figsize=(4, 3)) # create a dimension plot.
plt.bar(["H", "W", "C"], shape_b9, color=["gray", "gray", "orange"]) # show full tensor dimensions.
plt.title("Basic 9: output tensor dimensions") # title the plot.
plt.ylabel("size") # label size axis.
plt.show() # display dimension bars.

▶ What you'll see: channel depth changes to 8 while height and width stay 32.

👀 Takeaway: output channels are chosen by filter count; spatial dimensions are chosen by kernel, padding, and stride.

### Basic 10 — Check shape compatibility for addition

**Goal.** Test whether two feature maps can be added, because residual shortcuts require identical spatial and channel dimensions. We build it in 2 steps.

In [ ]:
main_b10 = np.zeros((8, 8, 16)) # main branch feature tensor.
shortcut_b10 = np.zeros((8, 8, 16)) # matching shortcut branch tensor.
bad_shortcut_b10 = np.zeros((4, 4, 16)) # shortcut after an accidental stride.
print("main shape:", main_b10.shape) # inspect main path.
print("shortcut shape:", shortcut_b10.shape) # inspect matching shortcut path.

▶ What you'll see: matching branches have the same height, width, and channel count.

In [ ]:
can_add_b10 = main_b10.shape == shortcut_b10.shape # check valid residual addition.
can_add_bad_b10 = main_b10.shape == bad_shortcut_b10.shape # check invalid residual addition.
print("can add matching shortcut:", can_add_b10) # inspect valid add condition.
print("can add strided shortcut:", can_add_bad_b10) # inspect invalid add condition.
assert can_add_b10 is True # identical shapes can be added elementwise.
assert can_add_bad_b10 is False # mismatched spatial size breaks elementwise addition.
plt.figure(figsize=(4, 3)) # create a compatibility plot.
plt.bar(["match", "mismatch"], [int(can_add_b10), int(can_add_bad_b10)], color=["seagreen", "crimson"]) # visualize shape checks.
plt.title("Basic 10: residual add shape check") # title the plot.
plt.ylim(0, 1.2) # keep boolean bars visible.
plt.show() # display the check.

▶ What you'll see: one wrong stride can make a shortcut structurally incompatible.

👀 Takeaway: CNN block design is partly math and partly strict tensor bookkeeping.

## 🟡 Easy

### Easy 1 — Build stride and padding into one convolution

**Goal.** Use one function to apply padding, stride, and local dot products, because production CNN layers combine these choices at once. We build it in 3 steps.

In [ ]:
x_e1 = np.arange(1, 26, dtype=float).reshape(5, 5) # define a 5x5 input.
k_e1 = np.array([[1., 0.], [0., -1.]]) # define a 2x2 detector.
pad_e1 = 1 # add one zero border.
stride_e1 = 2 # evaluate every other landing.
print("input shape:", x_e1.shape, "pad:", pad_e1, "stride:", stride_e1) # inspect layer choices.

▶ What you'll see: the layer will first enlarge the canvas, then sample it coarsely.

In [ ]:
y_e1 = conv2d(x_e1, k_e1, stride=stride_e1, pad=pad_e1) # compute padded strided convolution.
formula_e1 = out_size(5, 2, pad_e1, stride_e1) # compute expected spatial size.
print("output shape:", y_e1.shape, "formula length:", formula_e1) # compare code and formula.
assert y_e1.shape == (3, 3) # floor((5+2-2)/2)+1 = 3.

In [ ]:
print("output:\n", y_e1) # inspect actual responses.
show_map(y_e1, "Easy 1: padded stride-2 convolution") # visualize the combined effect.

▶ What you'll see: the output is 3×3 because padding created more legal starts before stride skipped positions.

👀 Takeaway: padding and stride interact through the same landing-position formula.

### Easy 2 — Compare valid and same-style padding

**Goal.** Show how padding changes feature-map size, because unpadded convolutions shrink maps while odd-kernel same-style padding preserves them. We build it in 3 steps.

In [ ]:
x_e2 = np.arange(1, 26, dtype=float).reshape(5, 5) # define a 5x5 input.
k_e2 = np.ones((3, 3)) # use a 3x3 sum kernel.
y_valid_e2 = conv2d(x_e2, k_e2, stride=1, pad=0) # no padding.
y_same_e2 = conv2d(x_e2, k_e2, stride=1, pad=1) # one-pixel padding for 3x3 kernel.
print("valid shape:", y_valid_e2.shape, "same-style shape:", y_same_e2.shape) # inspect sizes.

▶ What you'll see: valid convolution shrinks to 3×3 while same-style padding keeps 5×5.

In [ ]:
assert y_valid_e2.shape == (3, 3) # 5-3+1.
assert y_same_e2.shape == (5, 5) # (5+2-3)+1.
center_e2 = y_same_e2[2, 2] # center sees a full 3x3 original patch.
corner_e2 = y_same_e2[0, 0] # corner sees zeros in the padded window.
print("center sum:", center_e2, "corner sum:", corner_e2) # compare border and interior responses.
assert center_e2 == 117.0 # sum of center 3x3 block.

In [ ]:
plt.figure(figsize=(6, 2.7)) # create side-by-side maps.
plt.subplot(1, 2, 1); plt.imshow(y_valid_e2, cmap="viridis"); plt.title("valid 3x3") # plot valid output.
plt.subplot(1, 2, 2); plt.imshow(y_same_e2, cmap="viridis"); plt.title("pad 1 output") # plot padded output.
plt.show() # display both maps.

▶ What you'll see: padding preserves grid size, but border values differ because zero padding changes the window contents.

👀 Takeaway: same-style padding is a shape choice, not a promise that border responses behave like interior responses.

### Easy 3 — Pool after convolution

**Goal.** Build a small conv→pool pipeline, because CNN blocks often detect locally and then reduce resolution. We build it in 3 steps.

In [ ]:
x_e3 = np.arange(1, 26, dtype=float).reshape(5, 5) # define a toy image.
k_e3 = np.array([[1., 1.], [1., 1.]]) # sum 2x2 local evidence.
conv_e3 = conv2d(x_e3, k_e3, stride=1, pad=0) # compute dense local sums.
print("conv shape:", conv_e3.shape) # inspect convolution output size.

▶ What you'll see: the 5×5 input becomes a 4×4 activation map.

In [ ]:
pool_e3 = pool2d(conv_e3, size=2, stride=2, mode="max") # keep strongest activation in each 2x2 region.
print("pooled shape:", pool_e3.shape) # inspect downsampled size.
print("pooled values:\n", pool_e3) # inspect retained peaks.
assert pool_e3.shape == (2, 2) # 4x4 pooled by 2x2 stride 2 gives 2x2.
assert pool_e3[1, 1] == 88.0 # bottom-right pooled value from largest local sums.

In [ ]:
plt.figure(figsize=(6, 2.7)) # create pipeline visualization.
plt.subplot(1, 2, 1); plt.imshow(conv_e3, cmap="magma"); plt.title("conv activations") # plot activations.
plt.subplot(1, 2, 2); plt.imshow(pool_e3, cmap="magma"); plt.title("max pooled") # plot pooled map.
plt.show() # display both stages.

▶ What you'll see: pooling keeps the largest activation from each quadrant of the convolution map.

👀 Takeaway: conv→pool trades spatial detail for a smaller map of strong local evidence.

### Easy 4 — Compute layer parameters and activations

**Goal.** Separate parameter count from activation count, because filters decide weights while output tensor shape decides memory. We build it in 3 steps.

In [ ]:
h_e4, w_e4, cin_e4 = 32, 32, 3 # define input tensor dimensions.
kh_e4, kw_e4, cout_e4 = 3, 3, 8 # define convolution filter dimensions and count.
pad_e4, stride_e4 = 1, 1 # choose shape-preserving spatial settings.
oh_e4 = out_size(h_e4, kh_e4, pad_e4, stride_e4) # output height.
ow_e4 = out_size(w_e4, kw_e4, pad_e4, stride_e4) # output width.
print("output spatial:", (oh_e4, ow_e4)) # inspect spatial size.

▶ What you'll see: the layer keeps a 32×32 grid.

In [ ]:
params_e4 = kh_e4 * kw_e4 * cin_e4 * cout_e4 # weights without biases.
activations_e4 = oh_e4 * ow_e4 * cout_e4 # output values stored for one image.
print("parameters:", params_e4, "output activations:", activations_e4) # compare model size and data size.
assert params_e4 == 216 # verify weight count.
assert activations_e4 == 8192 # 32*32*8 output values.

In [ ]:
plt.figure(figsize=(4, 3)) # create a count comparison plot.
plt.bar(["weights", "activations"], [params_e4, activations_e4], color=["orange", "steelblue"]) # show counts.
plt.title("Easy 4: weights vs activations") # title the plot.
plt.ylabel("count") # label count scale.
plt.show() # display comparison.

▶ What you'll see: a small filter bank can produce many activation values over the spatial grid.

👀 Takeaway: convolution shares parameters spatially, so activation memory can dwarf parameter count.

### Easy 5 — Diagnose a residual-shape mismatch

**Goal.** Detect why two CNN paths cannot be added, because residual additions require identical tensors. We build it in 3 steps.

In [ ]:
input_shape_e5 = (16, 16, 8) # incoming feature map.
main_spatial_e5 = out_size(16, 3, p=1, s=2) # main path uses stride 2.
main_shape_e5 = (main_spatial_e5, main_spatial_e5, 16) # main path also changes channels.
shortcut_shape_e5 = input_shape_e5 # shortcut leaves the tensor unchanged.
print("main shape:", main_shape_e5) # inspect main path output.
print("shortcut shape:", shortcut_shape_e5) # inspect shortcut output.

▶ What you'll see: the main path is 8×8×16 while the shortcut remains 16×16×8.

In [ ]:
matches_e5 = main_shape_e5 == shortcut_shape_e5 # check elementwise-add compatibility.
projection_shape_e5 = main_shape_e5 # a projection shortcut would be designed to match the main path.
projection_matches_e5 = projection_shape_e5 == main_shape_e5 # check projected add compatibility.
print("raw shortcut add works:", matches_e5) # inspect failure.
print("projection add works:", projection_matches_e5) # inspect fix.
assert matches_e5 is False # mismatched spatial and channel dimensions cannot be added.
assert projection_matches_e5 is True # matching projection can be added.

In [ ]:
plt.figure(figsize=(5, 3)) # create a shape diagnostic plot.
plt.bar(["main H", "shortcut H", "main C", "shortcut C"], [8, 16, 16, 8], color=["teal", "red", "teal", "red"]) # compare mismatched dimensions.
plt.title("Easy 5: residual path mismatch") # title the plot.
plt.ylabel("dimension size") # label dimension scale.
plt.show() # display mismatch bars.

▶ What you'll see: both spatial size and channel count differ, so addition would be structurally invalid.

👀 Takeaway: downsampling or changing channels on one path requires a matching projection on the other path.

## 🔴 Advanced

### Advanced 1 — Sweep stride and track output size

**Goal.** Measure how stride changes output size and coverage, because larger strides reduce compute but skip more possible starts. We build it in 4 steps.

In [ ]:
n_a1 = 15 # input length.
k_a1 = 3 # kernel size.
p_a1 = 1 # padding.
strides_a1 = np.array([1, 2, 3, 4]) # stride values to compare.
sizes_a1 = np.array([out_size(n_a1, k_a1, p_a1, s_a1) for s_a1 in strides_a1]) # compute output lengths.
print("output sizes:", sizes_a1) # inspect shape shrinkage.

▶ What you'll see: larger stride values produce fewer output cells.

In [ ]:
landing_counts_a1 = [] # store number of starts for each stride.
for s_a1 in strides_a1: # inspect starts for each stride.
    last_a1 = n_a1 + 2 * p_a1 - k_a1 # last legal start on the padded line.
    starts_a1 = np.arange(0, last_a1 + 1, s_a1) # actual landings.
    landing_counts_a1.append(len(starts_a1)) # save count.
print("landing counts:", landing_counts_a1) # compare with formula sizes.
assert np.array_equal(sizes_a1, np.array(landing_counts_a1)) # formula equals counted starts.

In [ ]:
compute_ratio_a1 = (sizes_a1 ** 2) / (sizes_a1[0] ** 2) # approximate 2-D output-cell ratio versus stride 1.
print("relative output cells:", np.round(compute_ratio_a1, 3)) # inspect compute reduction.
assert compute_ratio_a1[0] == 1.0 # stride 1 baseline.

In [ ]:
plt.figure(figsize=(5, 3)) # create stride sweep plot.
plt.plot(strides_a1, sizes_a1, marker="o", label="output length") # plot shape shrinkage.
plt.plot(strides_a1, compute_ratio_a1 * sizes_a1[0], marker="s", label="scaled cell ratio") # plot compute proxy on comparable scale.
plt.title("Advanced 1: stride sweep") # title the plot.
plt.xlabel("stride") # label stride axis.
plt.legend() # show curves.
plt.show() # display the sweep.

▶ What you'll see: stride reduces both spatial size and the approximate number of convolution responses.

👀 Takeaway: stride is a compute-resolution knob with direct geometric consequences.

### Advanced 2 — Find padding that preserves size

**Goal.** Search padding values for different kernel sizes, because same-size padding is automatic only for certain kernel/stride choices. We build it in 4 steps.

In [ ]:
n_a2 = 7 # input size.
kernels_a2 = np.array([2, 3, 4, 5]) # compare even and odd kernels.
pads_a2 = np.arange(0, 5) # candidate padding values.
print("kernels:", kernels_a2, "candidate pads:", pads_a2) # inspect search grid.

▶ What you'll see: the sweep includes even kernels, which cannot preserve size with symmetric integer padding at stride 1.

In [ ]:
same_pads_a2 = {} # map each kernel to padding values that preserve size.
for k_a2 in kernels_a2: # loop over kernel sizes.
    hits_a2 = [int(p_a2) for p_a2 in pads_a2 if out_size(n_a2, int(k_a2), int(p_a2), 1) == n_a2] # find preserving pads.
    same_pads_a2[int(k_a2)] = hits_a2 # store hits.
print("same-size pads:", same_pads_a2) # inspect which kernels work.
assert same_pads_a2[3] == [1] # 3x3 needs pad 1.
assert same_pads_a2[5] == [2] # 5x5 needs pad 2.
assert same_pads_a2[2] == [] # symmetric integer pad cannot preserve size for k=2.

In [ ]:
out_table_a2 = np.array([[out_size(n_a2, int(k_a2), int(p_a2), 1) for p_a2 in pads_a2] for k_a2 in kernels_a2]) # compute all output sizes.
print("rows=kernels, cols=pads:\n", out_table_a2) # inspect the shape table.

In [ ]:
plt.figure(figsize=(5, 3)) # create output-size heatmap.
plt.imshow(out_table_a2, cmap="viridis", aspect="auto") # show output size for every k,p pair.
plt.colorbar(label="output size") # add numeric scale.
plt.xticks(range(len(pads_a2)), pads_a2) # label pad candidates.
plt.yticks(range(len(kernels_a2)), kernels_a2) # label kernel sizes.
plt.xlabel("padding p") # label x-axis.
plt.ylabel("kernel k") # label y-axis.
plt.title("Advanced 2: same-size padding search") # title the heatmap.
plt.show() # display the table.

▶ What you'll see: odd kernels have a clean same-size padding; even kernels jump around the target size.

👀 Takeaway: padding preserves size only when the formula says it does; do not assume every pad value is “same.”

### Advanced 3 — Compare pooling choices on localization

**Goal.** Show how max and average pooling respond to a shifted peak, because pooling creates partial translation tolerance while losing exact location. We build it in 4 steps.

In [ ]:
map_a3 = np.zeros((6, 6)) # create an empty activation map.
map_a3[1, 1] = 10.0 # place a strong feature in the upper-left pooling region.
shifted_a3 = np.zeros((6, 6)) # create a shifted version.
shifted_a3[2, 2] = 10.0 # move the feature into a neighboring pooling region boundary.
print("peak positions:", np.argwhere(map_a3 == 10)[0], np.argwhere(shifted_a3 == 10)[0]) # inspect locations.

▶ What you'll see: the same strong activation moves from one local window to another.

In [ ]:
max_map_a3 = pool2d(map_a3, size=2, stride=2, mode="max") # max pool original.
max_shift_a3 = pool2d(shifted_a3, size=2, stride=2, mode="max") # max pool shifted.
avg_map_a3 = pool2d(map_a3, size=2, stride=2, mode="avg") # average pool original.
avg_shift_a3 = pool2d(shifted_a3, size=2, stride=2, mode="avg") # average pool shifted.
print("max original:\n", max_map_a3) # inspect max pooled original.
print("max shifted:\n", max_shift_a3) # inspect max pooled shifted.

In [ ]:
assert np.max(max_map_a3) == 10.0 # max preserves peak magnitude.
assert np.max(avg_map_a3) == 2.5 # average spreads a single peak over a 2x2 window.
changed_cells_a3 = np.sum(max_map_a3 != max_shift_a3) # count changed pooled cells after shift.
print("changed max-pooled cells:", changed_cells_a3) # inspect localization sensitivity.
assert changed_cells_a3 == 2 # one pooled cell loses the peak and another gains it.

In [ ]:
plt.figure(figsize=(7, 3)) # create original-vs-shifted pooled maps.
plt.subplot(1, 2, 1); plt.imshow(max_map_a3, cmap="magma", vmin=0, vmax=10); plt.title("max pool original") # plot original pooled peak.
plt.subplot(1, 2, 2); plt.imshow(max_shift_a3, cmap="magma", vmin=0, vmax=10); plt.title("max pool shifted") # plot shifted pooled peak.
plt.show() # display both maps.

▶ What you'll see: max pooling preserves peak strength but may move which pooled cell contains it.

👀 Takeaway: pooling gives tolerance inside a window, not perfect invariance across window boundaries.

### Advanced 4 — Implement multi-channel convolution at one location

**Goal.** Compute one RGB convolution response from scratch, because each output filter combines evidence across all input channels. We build it in 4 steps.

In [ ]:
patch_a4 = np.arange(1, 28, dtype=float).reshape(3, 3, 3) # create one 3x3 RGB-like patch.
filter_a4 = np.ones((3, 3, 3)) # use a filter that sums every spatial and channel value.
print("patch shape:", patch_a4.shape, "filter shape:", filter_a4.shape) # inspect channel-aware shapes.

▶ What you'll see: both patch and filter have height, width, and channel dimensions.

In [ ]:
channel_sums_a4 = np.array([np.sum(patch_a4[:, :, c_a4] * filter_a4[:, :, c_a4]) for c_a4 in range(3)]) # sum each channel's contribution.
response_a4 = float(np.sum(channel_sums_a4)) # combine channels into one output-channel response.
print("channel contributions:", channel_sums_a4) # inspect per-channel evidence.
print("response:", response_a4) # inspect final scalar response.
assert response_a4 == 378.0 # sum numbers 1 through 27.

In [ ]:
filters_a4 = np.stack([filter_a4, -filter_a4], axis=0) # create two output filters.
responses_a4 = np.array([np.sum(patch_a4 * filters_a4[o_a4]) for o_a4 in range(filters_a4.shape[0])]) # compute one response per output channel.
print("two output-channel responses:", responses_a4) # inspect filter-bank output at one location.
assert np.array_equal(responses_a4, np.array([378., -378.])) # opposite filters produce opposite responses.

In [ ]:
plt.figure(figsize=(4, 3)) # create channel contribution plot.
plt.bar(["channel 0", "channel 1", "channel 2"], channel_sums_a4, color="steelblue") # visualize per-channel sums.
plt.title("Advanced 4: multi-channel dot product") # title the plot.
plt.ylabel("contribution") # label contribution scale.
plt.show() # display channel contributions.

▶ What you'll see: one output value is the sum of spatial evidence across all input channels.

👀 Takeaway: a convolutional filter is 3-D over height, width, and input channels; multiple filters create output channels.

### Advanced 5 — Design a mini CNN shape schedule

**Goal.** Track a sequence of convolution and pooling layers, because real CNNs are planned as resolution schedules rather than isolated formulas. We build it in 5 steps.

In [ ]:
shape_a5 = (32, 32, 3) # start with a 32x32 RGB image.
layers_a5 = [] # store the shape after each block.
print("start shape:", shape_a5) # inspect input tensor.

▶ What you'll see: the network starts at full image resolution with three channels.

In [ ]:
h_a5, w_a5, c_a5 = shape_a5 # unpack current tensor shape.
h_a5 = out_size(h_a5, 3, p=1, s=1) # conv1 preserves height.
w_a5 = out_size(w_a5, 3, p=1, s=1) # conv1 preserves width.
c_a5 = 8 # conv1 uses eight filters.
layers_a5.append(("conv3 pad1 stride1", h_a5, w_a5, c_a5)) # record conv1 output.
print("after conv1:", layers_a5[-1]) # inspect first layer shape.
assert (h_a5, w_a5, c_a5) == (32, 32, 8) # verify conv1 output.

In [ ]:
h_a5 = out_size(h_a5, 2, p=0, s=2) # 2x2 pool halves height.
w_a5 = out_size(w_a5, 2, p=0, s=2) # 2x2 pool halves width.
layers_a5.append(("maxpool2 stride2", h_a5, w_a5, c_a5)) # record pool output.
print("after pool:", layers_a5[-1]) # inspect pooled shape.
assert (h_a5, w_a5, c_a5) == (16, 16, 8) # verify downsampled shape.

In [ ]:
h_a5 = out_size(h_a5, 3, p=1, s=2) # strided conv halves height again.
w_a5 = out_size(w_a5, 3, p=1, s=2) # strided conv halves width again.
c_a5 = 16 # second conv uses sixteen filters.
layers_a5.append(("conv3 pad1 stride2", h_a5, w_a5, c_a5)) # record final block output.
print("after conv2:", layers_a5[-1]) # inspect final feature shape.
assert (h_a5, w_a5, c_a5) == (8, 8, 16) # verify final shape.

In [ ]:
labels_a5 = [row_a5[0] for row_a5 in layers_a5] # gather layer names.
spatial_cells_a5 = [row_a5[1] * row_a5[2] for row_a5 in layers_a5] # compute spatial cell counts.
channels_a5 = [row_a5[3] for row_a5 in layers_a5] # gather channel counts.
print("spatial cells:", spatial_cells_a5) # inspect resolution schedule.
print("channels:", channels_a5) # inspect depth schedule.
plt.figure(figsize=(6, 3)) # create schedule plot.
plt.plot(spatial_cells_a5, marker="o", label="H×W cells") # plot spatial resolution.
plt.plot(channels_a5, marker="s", label="channels") # plot channel depth.
plt.xticks(range(len(labels_a5)), ["conv1", "pool", "conv2"]) # label stages.
plt.title("Advanced 5: CNN shape schedule") # title the plot.
plt.legend() # show curves.
plt.show() # display schedule.

▶ What you'll see: spatial resolution falls from 32×32 to 8×8 while channel depth rises from 3 to 16.

👀 Takeaway: CNNs usually spend early layers on fine resolution and later layers on deeper, coarser semantic maps.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Stride, padding, and pooling are the small shape decisions that decide how much detail a CNN keeps, moves past, or throws away.

Stride decides which windows are evaluated, padding changes border coverage, and pooling summarizes local evidence without learning new weights. Most CNN shape bugs come from forgetting the floor or the channel dimension.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (10, 3)


## The concept, built once
$$n_{out}=\left\lfloor\frac{n+2p-k}{s}\right\rfloor+1$$

We implement the small reusable method for the lesson and assert the exact worked numbers from the plan.

In [ ]:

def output_size(n, k, padding=0, stride=1):
    return int(np.floor((n + 2 * padding - k) / stride) + 1)


def max_pool2x2(x):
    arr = np.asarray(x)
    out = np.zeros((arr.shape[0] // 2, arr.shape[1] // 2))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            block = arr[2 * i:2 * i + 2, 2 * j:2 * j + 2]
            out[i, j] = block.max()
    return out


def cnn_block(n, k, padding, stride, in_channels=3, out_channels=8):
    size = output_size(n, k, padding, stride)
    weights = k * k * in_channels * out_channels
    return size, weights

n_stride, _ = cnn_block(5, 2, 0, 2)
n_pad, _ = cnn_block(5, 2, 1, 1)
activation = np.array([[1, 6, 2, 5], [3, 4, 1, 0], [2, 1, 9, 8], [0, 1, 3, 7]])
pooled = max_pool2x2(activation)
_, weights = cnn_block(8, 3, 1, 1)
assert n_stride == 2
assert n_pad == 6
assert np.array_equal(pooled, np.array([[6, 5], [2, 9]]))
assert weights == 216

print("stride output", n_stride)
print("padded output", n_pad)
print("pooled")
print(pooled)
print("RGB 3x3x8 weights", weights)


The method above is now reusable. Next we wrap it as a featurizer so the same idea can be tested across all five dataset rungs.

In [ ]:

def featurize(img):
    pooled = max_pool2x2(img)
    padded = np.pad(img, 1)
    center = padded[1:7, 1:7].ravel()
    return np.concatenate([pooled.ravel(), center, img.mean(axis=0), img.mean(axis=1)])


def feature_map_for_display(img):
    return max_pool2x2(img)



## The dataset ladder
We reuse the shared D1-D5 classification ladder. Each rung returns 8x8 images and labels, so the same featurizer can be evaluated with held-out accuracy as complexity rises.


In [ ]:
"""
F6 (Vision) shared dataset ladder — D1..D5 of rising complexity, CPU-only and run-all-safe.

This is the canonical ladder inlined into the classification-style Part-7 notebooks. Every
rung returns (X, y) with X shape (n, 8, 8) float in [0, 1] and integer labels y, so one
featurizer + classifier can run unchanged across all five rungs (the "watch it scale" story).

D4/D5 load real MNIST / CIFAR-10 via torchvision when the download is available (as in Colab),
offline they fall back to a harder synthetic set so run-all never fails. Code is written one
statement per line for readability.
"""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _resize_to_8x8(img):
    """Nearest-neighbour resize of a 2-D array to 8x8 (no SciPy dependency)."""
    h, w = img.shape
    rows = (np.linspace(0, h - 1, 8)).round().astype(int)
    cols = (np.linspace(0, w - 1, 8)).round().astype(int)
    return img[np.ix_(rows, cols)]


def _normalize(x):
    """Scale an array into [0, 1], a flat array becomes all zeros."""
    x = x.astype(float)
    lo = x.min()
    hi = x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def d1_hand_patches():
    """D1 — hand-built 4x4 patches, 2 classes: a vertical line (col 1) vs a horizontal line (row 1).

    Fixed positions with light jitter, so the two classes are cleanly separable and the
    mechanism is fully visible — the easy first rung.
    """
    rng = np.random.default_rng(0)
    images = []
    labels = []
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[:, 1] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(0)
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[1, :] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(1)
    return np.array(images), np.array(labels)


def d2_synthetic_shapes():
    """D2 — clean synthetic shapes on an 8x8 grid, 2 classes (square vs disc)."""
    rng = np.random.default_rng(1)
    yy, xx = np.mgrid[0:8, 0:8]
    images = []
    labels = []
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        img[2:6, 2:6] = 0.9
        images.append(img)
        labels.append(0)
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        disc = (xx - 3.5) ** 2 + (yy - 3.5) ** 2 <= 4.0
        img[disc] = 0.9
        images.append(img)
        labels.append(1)
    return np.array(images), np.array(labels)


def d3_sklearn_digits():
    """D3 — real sklearn digits (native 8x8), 4 classes for a fast, honest multi-class rung."""
    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    X = digits.images[keep]
    y = digits.target[keep]
    X = np.array([_normalize(img) for img in X])
    return X, y


def _synthetic_textured(n_per_class, n_classes, noise, seed):
    """A harder synthetic fallback: textured class prototypes at 8x8 with noise."""
    rng = np.random.default_rng(seed)
    protos = [rng.uniform(0.0, 1.0, size=(8, 8)) for _ in range(n_classes)]
    images = []
    labels = []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            img = protos[cls] + rng.normal(0.0, noise, size=(8, 8))
            images.append(_normalize(img))
            labels.append(cls)
    return np.array(images), np.array(labels)


def _call_with_timeout(fn, seconds):
    """Run fn() but abort with TimeoutError after `seconds` (guards slow/hanging downloads)."""
    import signal

    def _raise(signum, frame):
        raise TimeoutError("download timed out")

    old = signal.signal(signal.SIGALRM, _raise)
    signal.alarm(seconds)
    try:
        return fn()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def _load_mnist_gray(classes, n_per_class, seed, shift=False, noise=0.0):
    """Load MNIST via torchvision, grayscale + resize to 8x8, subsample. Raises on failure.

    MNIST is a small (~11 MB) real dataset. CIFAR-10 is deliberately avoided (a 170 MB
    download breaks run-all-safety), the harder D5 rung instead shifts and noises MNIST.
    """
    import torchvision

    ds = torchvision.datasets.MNIST(root="./data", train=True, download=True)
    rng = np.random.default_rng(seed)
    targets = np.array(ds.targets)
    images = []
    labels = []
    for cls in classes:
        idx = np.where(targets == cls)[0][:n_per_class]
        for i in idx:
            arr = np.asarray(ds[int(i)][0], dtype=float)
            small = _resize_to_8x8(arr)
            if shift:
                small = np.roll(small, rng.integers(-1, 2), axis=0)
                small = np.roll(small, rng.integers(-1, 2), axis=1)
            if noise:
                small = small + rng.normal(0.0, noise * 255.0, size=(8, 8))
            images.append(_normalize(small))
            labels.append(cls)
    return np.array(images), np.array(labels)


def d4_mnist_or_fallback():
    """D4 — real MNIST (4 clean classes) when downloadable, else a harder synthetic set."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3], 60, seed=4), 30)
        return (X, y), "MNIST (real)"
    except Exception:
        return _synthetic_textured(60, 4, noise=0.35, seed=4), "synthetic (offline fallback)"


def d5_mnist_hard_or_fallback():
    """D5 — real MNIST, more classes with shift + noise (distribution shift), else hardest synthetic."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3, 4, 5], 60, seed=5, shift=True, noise=0.12), 30)
        return (X, y), "MNIST shifted+noisy (real, harder)"
    except Exception:
        return _synthetic_textured(60, 6, noise=0.6, seed=5), "synthetic (offline fallback)"


def load_ladder():
    """Return the five rungs as a list of (name, X, y). D4/D5 note whether real data loaded."""
    rungs = []
    rungs.append(("D1 hand patches", *d1_hand_patches()))
    rungs.append(("D2 synthetic shapes", *d2_synthetic_shapes()))
    rungs.append(("D3 sklearn digits", *d3_sklearn_digits()))
    (x4, y4), tag4 = d4_mnist_or_fallback()
    rungs.append((f"D4 {tag4}", x4, y4))
    (x5, y5), tag5 = d5_mnist_hard_or_fallback()
    rungs.append((f"D5 {tag5}", x5, y5))
    return rungs


def accuracy_with(featurize, X, y):
    """Map each image through featurize, train logistic regression, return held-out accuracy."""
    feats = np.array([featurize(img) for img in X])
    x_tr, x_te, y_tr, y_te = train_test_split(feats, y, test_size=0.4, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.score(x_te, y_te)

In [ ]:

rungs = load_ladder()

for name, X, y in rungs:
    classes = sorted(set(y.tolist()))
    print(f"{name:38s} shape={X.shape} classes={classes}")

fig, axes = plt.subplots(1, len(rungs), figsize=(12, 2.4))

for ax, (name, X, y) in zip(axes, rungs):
    ax.imshow(X[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(name.split()[0])
    ax.axis("off")

plt.tight_layout()
plt.show()


## Run the same method across D1-D5
The classifier is intentionally simple; changes in accuracy come from the feature representation and the data complexity.

In [ ]:

accuracies = []
feature_examples = []

for name, X, y in rungs:
    acc = accuracy_with(featurize, X, y)
    accuracies.append(acc)
    feature_examples.append(feature_map_for_display(X[0]))
    print(f"{name:38s} accuracy={acc:.3f}")


In [ ]:

def majority_baseline(y):
    values, counts = np.unique(y, return_counts=True)
    return counts.max() / counts.sum()

baseline_name, baseline_X, baseline_y = rungs[-1]
print("D5 majority baseline", round(majority_baseline(baseline_y), 3))


## Results visualization
The first figure shows one feature-map panel per rung; the second tracks accuracy against ladder complexity.

In [ ]:

fig, axes = plt.subplots(1, len(feature_examples), figsize=(12, 2.4))

for ax, fmap, (name, X, y) in zip(axes, feature_examples, rungs):
    ax.imshow(fmap, cmap="magma")
    ax.set_title(name.split()[0])
    ax.axis("off")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
xs = np.arange(1, len(accuracies) + 1)
ax.plot(xs, accuracies, marker="o")
ax.set_xticks(xs)
ax.set_xticklabels([f"D{i}" for i in xs])
ax.set_ylim(0, 1.05)
ax.set_ylabel("accuracy")
ax.set_title("Accuracy vs. ladder rung")
ax.grid(True, alpha=0.3)
plt.show()



## Pitfall on D5
Dropping the floor predicts impossible fractional cells and can break shape alignment in later blocks.


In [ ]:

wrong_size = (5 + 0 - 2) / 2 + 1
fixed_size = output_size(5, 2, 0, 2)
name, X, y = rungs[-1]
wrong_feature_length = int(np.ceil(wrong_size)) ** 2
fixed_feature_length = fixed_size ** 2
print("wrong no-floor size", wrong_size)
print("wrong implied cells", wrong_feature_length)
print("fixed integer size", fixed_size)
print("fixed cells", fixed_feature_length)



## Evaluate it + Practice
- Metric: held-out accuracy from the same logistic-regression probe on every rung; the no-skill baseline is majority-class accuracy.
- Sanity check: D1 should be easy because the visual cue is deliberately visible.
- Ablation: turn the key feature off or coarsen it and confirm accuracy or localization drops.
- Failure signals: shape mismatches, feature maps with unexpected orientation, or a D5 score that changes wildly under a deterministic seed.

Practice prompts:
1. Change one constant in the D1 worked example and update the assert.



2. Replace the featurizer with raw pixels and compare the accuracy curve.



3. Make the pitfall more severe, then design a smaller fix.
